# 5단계: 오경보 소리까지 학습에 넣어 다시 학습 + 연속 3시간 최종 시험

04에서 오경보가 난 소리(하품, 가족 대화, 쇼츠, 감탄사, 베개 두드림 등)는 **정상이었습니다.** 이번에는
1. `6_own`의 **학습용 + 시험용 녹음을 전부 정상 소리로** 학습에 넣고,
2. 04의 새 모델이 오경보를 낸 프레임에는 **가중치를 더 줘서**(어려운 정상 소리) 다시 학습합니다.
3. 파일 이름에 **`연속` 또는 `3시간`**이 들어간 녹음은 학습에 쓰지 않고 **최종 시험**으로만 씁니다. 없으면 이 단계는 건너뜁니다.
4. 폴더 이름에 **`응급`**이 들어간 녹음(위급 연기)도 학습에서 제외합니다.

런타임은 **CPU로 충분**합니다. 04에서 만든 임베딩 캐시(`own_embeddings_v1.pkl`)가 있으면 새 파일만 처리합니다.
위에서부터 순서대로 실행하고, **5번, 6번, 7번 항목의 출력**을 복사해서 알려 주세요.

## 0. 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 파일 찾기

In [ ]:
import glob, os, re, subprocess, pickle, time, csv, unicodedata
import numpy as np
import pandas as pd

def find(name):
    hits = glob.glob(f"/content/drive/MyDrive/**/{name}", recursive=True)
    assert hits, f"{name}을 못 찾았어요. 드라이브에 올렸는지 확인하세요."
    return hits[0]

OWN = find("6_own")
EMB = find("embeddings_v2.npz")
OUT_DIR = os.path.dirname(EMB)
files = sorted(glob.glob(f"{OWN}/*/*.m4a") + glob.glob(f"{OWN}/*/*.wav") + glob.glob(f"{OWN}/*/*.mp3"))
print("녹음 파일", len(files), "개 / 작업 폴더:", OUT_DIR)

## 2. YAMNet 불러오기

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub

yamnet = hub.load("https://tfhub.dev/google/yamnet/1")
with tf.io.gfile.GFile(yamnet.class_map_path().numpy()) as f:
    names = [row["display_name"] for row in csv.DictReader(f)]
SPEECH = names.index("Speech")
SCREAM = [names.index(n) for n in ("Screaming", "Shout", "Yell")]
print(len(names), "클래스")

## 3. 새 녹음만 변환·임베딩
이미 처리한 파일은 캐시를 씁니다. 연속 3시간 파일이 있으면 여기서 처리됩니다. (3시간이면 몇 분)

In [ ]:
FFMPEG = "ffmpeg"
SR, HOP, WIN = 16000, 7680, 15600
SEG = HOP * 600          # 288초씩 나눠서 처리 (프레임 격자는 이어짐)

def probe_channels(path):
    r = subprocess.run([FFMPEG, "-hide_banner", "-i", path], capture_output=True, text=True, errors="replace")
    m = re.search(r"Audio: .*?, \d+ Hz, ([^,\n]+)", r.stderr)
    return 1 if (m and m.group(1).strip().startswith(("mono", "1 channel"))) else 2

def load_mono(path):
    ch = probe_channels(path)
    raw = subprocess.run([FFMPEG, "-v", "error", "-i", path, "-vn", "-ac", str(ch), "-ar", str(SR), "-f", "s16le", "-"],
                         capture_output=True).stdout
    x = np.frombuffer(raw[: len(raw) // (2 * ch) * (2 * ch)], dtype=np.int16).reshape(-1, ch)
    if ch == 1:
        return x[:, 0].copy(), "모노", None
    a, b = x[::10, 0].astype(np.float64), x[::10, 1].astype(np.float64)
    corr = float(np.corrcoef(a, b)[0, 1]) if a.std() > 0 and b.std() > 0 else 1.0
    if corr < -0.3:
        return x[:, int(b.std() > a.std())].copy(), "한 채널(위상 반대)", corr
    return x.astype(np.float32).mean(axis=1).astype(np.int16), "두 채널 평균", corr

def embed_stream(x):
    embs, sp, sc, db, ts = [], [], [], [], []
    n = len(x)
    for start in range(0, n, SEG):
        seg = x[start:start + SEG + (WIN - HOP)]
        if len(seg) < WIN:
            break
        w = seg.astype(np.float32) / 32768
        scores, emb, _ = yamnet(tf.constant(w))
        scores, emb = scores.numpy(), emb.numpy()
        last = start + SEG + (WIN - HOP) >= n
        nb = len(w) // HOP
        bp = (w[: nb * HOP].reshape(nb, HOP) ** 2).mean(axis=1)
        fdb = 10 * np.log10(((bp[:-1] + bp[1:]) / 2 if nb > 1 else bp) + 1e-12)
        k = min(len(emb) if last else 600, len(emb), len(fdb))
        embs.append(emb[:k].astype(np.float16))
        sp.append(scores[:k, SPEECH].astype(np.float16))
        sc.append(scores[:k][:, SCREAM].max(axis=1).astype(np.float16))
        db.append(fdb[:k].astype(np.float32))
        ts.append(start / SR + np.arange(k) * 0.48)
    return tuple(np.concatenate(a) for a in (embs, sp, sc, db, ts))


PKL = f"{OUT_DIR}/own_embeddings_v1.pkl"
own = pickle.load(open(PKL, "rb")) if os.path.exists(PKL) else {}
nfc = lambda s: unicodedata.normalize("NFC", s)
for v in own.values():
    v["split"], v["name"] = nfc(v["split"]), nfc(v["name"])

t0 = time.time()
for f in files:
    if f in own:
        continue
    x, how, corr = load_mono(f)
    emb, sp, sc, db, ts = embed_stream(x)
    own[f] = dict(split=nfc(os.path.basename(os.path.dirname(f))), name=nfc(os.path.basename(f)), emb=emb, speech=sp,
                  scream=sc, db=db, t=ts, dur=len(x) / SR, how=how, corr=corr)
    print(f"새로 처리: {own[f]['name'][:30]} {own[f]['dur'] / 60:.1f}분 | {how} | 말소리 {float((sp > 0.5).mean()):.0%}", flush=True)
pickle.dump(own, open(PKL, "wb"))

is_C = lambda v: ("연속" in v["name"]) or ("3시간" in v["name"]) or ("장시간" in v["split"])
is_E = lambda v: "응급" in v["split"]          # 위급 연기 녹음 폴더는 학습에 쓰지 않음(시험용)
train_files = [f for f, v in own.items() if not is_C(v) and not is_E(v)]
C_files = [f for f, v in own.items() if is_C(v)]
print(f"학습에 쓸 녹음 {len(train_files)}개 ({sum(own[f]['dur'] for f in train_files) / 60:.0f}분)")
for f in C_files:
    v = own[f]
    print(f"최종 시험(C): {v['name']} {v['dur'] / 60:.0f}분 | 말소리 {float((v['speech'] > 0.5).mean()):.0%} "
          f"(약 {float((v['speech'] > 0.5).mean()) * v['dur'] / 60:.0f}분)")
n_E = sum(is_E(v) for v in own.values())
if n_E:
    print(f"응급 소리 폴더 녹음 {n_E}개는 학습에서 제외했습니다 (위급 재현율 시험용)")
if not C_files:
    print("최종 시험(C) 파일이 아직 없어요 → 학습과 저장까지만 합니다.")

## 4. AI Hub 임베딩과 평가 도구

In [ ]:
from tensorflow import keras

d = np.load(EMB)
X, S, clip_idx = d["X"].astype(np.float32), d["speech"].astype(np.float32), d["clip_idx"]
df = pd.read_csv(find("dataset_v2.csv"), encoding="utf-8-sig")
y_clip = (df["label"] == "urgent").astype(int).values
is_tr = (df["split"] == "train").values
starts = np.r_[0, np.flatnonzero(np.diff(clip_idx)) + 1]
ends = np.r_[starts[1:], len(clip_idx)]
va_urgent_idx = np.flatnonzero((~is_tr) & (y_clip == 1))
cls_val = df["class"].values[va_urgent_idx]
URG = ["10.낙상", "14.도움요청", "7.응급의료"]

def clip_scores(p, N):
    out = []
    for i in va_urgent_idx:
        q = p[starts[i]:ends[i]]
        out.append(np.lib.stride_tricks.sliding_window_view(q, N).min(axis=1).max() if len(q) >= N else 0.0)
    return np.array(out)

def alarm_idx(p, t, N, thr, cooldown=5.0):
    ok = (p >= thr).astype(int)
    if len(ok) < N:
        return []
    run = np.convolve(ok, np.ones(N, int), "valid") == N
    out, last = [], -1e9
    for i in np.flatnonzero(run) + (N - 1):
        if t[i] - last >= cooldown:
            out.append(int(i)); last = t[i]
    return out

ref_files = [f for f in train_files if "위급 장면" not in own[f]["name"]]
hours_ref = sum(own[f]["dur"] for f in ref_files) / 3600
hours_C = sum(own[f]["dur"] for f in C_files) / 3600

def evaluate(model, name):
    p_ai = model.predict(X, batch_size=4096, verbose=0).ravel()
    pf = {f: model.predict(own[f]["emb"].astype(np.float32), batch_size=4096, verbose=0).ravel() for f in own}
    rows = []
    for N in (1, 2, 3):
        sc = clip_scores(p_ai, N)
        for thr in (0.5, 0.8, 0.9, 0.95):
            r = {"모델": name, "N": N, "임계값": thr, "위급재현율": (sc >= thr).mean()}
            for c in URG:
                r[c] = (sc[cls_val == c] >= thr).mean()
            r["학습녹음시간당(참고)"] = sum(len(alarm_idx(pf[f], own[f]["t"], N, thr)) for f in ref_files) / hours_ref
            if C_files:
                n = sum(len(alarm_idx(pf[f], own[f]["t"], N, thr)) for f in C_files)
                r["C오경보(횟수)"], r["C시간당"] = n, n / hours_C
            rows.append(r)
    return pd.DataFrame(rows), pf

## 5. 오경보 프레임에 가중치를 주고 다시 학습
- 이전 모델(04에서 저장한 `classifier_mlp_v3_own`)이 확률 0.5 이상으로 울린 **정상 프레임**을 어려운 소리로 보고 `HARD_BOOST`배 가중합니다.
- 학습용·시험용 녹음은 약 10초 조각으로 나눠 정상 클립으로 넣습니다. (`OWN_BOOST`배)

In [ ]:
OWN_BOOST, HARD_BOOST, NORMAL_BOOST, SPEECH_MIN, CH = 3.0, 4.0, 3.0, 0.2, 20
try:
    prev = keras.models.load_model(find("classifier_mlp_v3_own.keras"))
except AssertionError as e:
    prev = None
    print("이전 모델이 없어 어려운 소리 가중치는 건너뜁니다:", e)

chunks, hard_chunks = [], []
for f in train_files:
    e = own[f]["emb"].astype(np.float32)
    ph = prev.predict(e, batch_size=4096, verbose=0).ravel() if prev is not None else np.zeros(len(e))
    for k in range(len(e) // CH):
        chunks.append(e[k * CH:(k + 1) * CH]); hard_chunks.append(ph[k * CH:(k + 1) * CH] >= 0.5)
n_own = len(chunks)
hard_f = np.concatenate(hard_chunks)
print(f"녹음 조각 {n_own}개 (약 10초씩), 어려운 정상 프레임 {int(hard_f.sum())}개 / {len(hard_f)}")

X_all = np.concatenate([X, np.concatenate(chunks)])
S_all = np.concatenate([S, np.ones(n_own * CH, np.float32)])
clip_all = np.concatenate([clip_idx, len(df) + np.repeat(np.arange(n_own), CH)])
y_all = np.r_[y_clip, np.zeros(n_own, int)]
tr_all = np.r_[is_tr, np.ones(n_own, bool)]
fr_all = np.bincount(clip_all)
boost = np.r_[np.ones(len(df)), np.full(n_own, OWN_BOOST)]

n_urg = int(y_all[tr_all].sum()); n_nor = int(tr_all.sum()) - n_urg
cw = {1: 0.5 * tr_all.sum() / n_urg, 0: NORMAL_BOOST * 0.5 * tr_all.sum() / n_nor}
w = np.array([cw[int(l)] for l in y_all])[clip_all] * boost[clip_all] / fr_all[clip_all]
w[len(X):] *= np.where(hard_f, HARD_BOOST, 1.0)
y_f = y_all[clip_all].astype(np.float32)
tr_f = tr_all[clip_all]
w[(y_f == 1) & (S_all < SPEECH_MIN)] = 0
w = (w / w[tr_f].mean()).astype(np.float32)

tf.keras.utils.set_random_seed(42)
model_new = keras.Sequential([
    keras.layers.Input(shape=(1024,)),
    keras.layers.Dense(256, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
])
model_new.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy")
es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
model_new.fit(X_all[tr_f], y_f[tr_f], sample_weight=w[tr_f],
              validation_data=(X_all[~tr_f], y_f[~tr_f], w[~tr_f]),
              epochs=40, batch_size=512, callbacks=[es], verbose=2)

## 6. 평가
- **위급재현율 / 낙상·도움요청·응급의료**: AI Hub Validation 위급 클립 중 알림이 울린 비율 (클래스별로도 확인)
- **학습녹음시간당(참고)**: 학습에 쓴 녹음에서의 오경보. 이미 학습한 소리라 **참고용**입니다.
- **C오경보 / C시간당**: 연속 3시간 최종 시험의 오경보 (학습에 쓰지 않은 소리). C 파일이 있을 때만 나옵니다.
- 모델 `이전(04)`은 학습 전, `새 모델`은 이번에 학습한 것입니다.

In [ ]:
pd.set_option("display.width", 220)
rows, probs = [], {}
if prev is not None:
    t_prev, probs["이전(04)"] = evaluate(prev, "이전(04)"); rows.append(t_prev)
t_new, probs["새 모델"] = evaluate(model_new, "새 모델"); rows.append(t_new)
print(pd.concat(rows, ignore_index=True).round(3).to_string(index=False))

## 7. 최종 시험(C)에서 울린 시각
`SHOW`의 (N, 임계값)에서 **새 모델**이 울린 시각입니다. 직접 들어 보고 무슨 소리였는지 알려 주세요. (C가 없으면 건너뜁니다)

In [ ]:
SHOW = (2, 0.9)
mmss = lambda s: f"{int(s) // 3600}:{int(s) % 3600 // 60:02d}:{int(s) % 60:02d}"
N0, THR0 = SHOW
if not C_files:
    print("최종 시험(C) 파일이 없어서 건너뜁니다.")
for f in C_files:
    v, p = own[f], probs["새 모델"][f]
    idx = alarm_idx(p, v["t"], N0, THR0)
    print(f"[{v['name']}] N={N0}, 임계값={THR0}: 알림 {len(idx)}회 / {v['dur'] / 3600:.2f}시간 (약 {len(idx) / (v['dur'] / 3600):.1f}회/시간)")
    for i in idx[:40]:
        j = i - (N0 - 1)
        print(f"  {mmss(v['t'][j])}  확률 {p[j:i + 1].min():.2f}  말소리 {float(v['speech'][j]):.2f}  음량 {v['db'][j]:.0f}dBFS")
    if len(idx) > 40:
        print(f"  ... 이하 {len(idx) - 40}회 생략")

## 8. 모델 저장

In [ ]:
model_new.save(f"{OUT_DIR}/classifier_mlp_v4_own_all.keras")
print("저장 완료:", f"{OUT_DIR}/classifier_mlp_v4_own_all.keras")